In [1]:
import json
import chromadb
from sentence_transformers import SentenceTransformer

# ---------------------------------------------------------
# 1. Initialize Models and Database
# ---------------------------------------------------------
print("Loading SentenceTransformer model...")
# Using a lightweight, fast model for semantic search
model = SentenceTransformer('all-MiniLM-L6-v2')

# Initialize an in-memory ChromaDB client
chroma_client = chromadb.Client()

# Create or get the collection where candidate vectors will live
collection = chroma_client.get_or_create_collection(name="candidate_profiles")

# ---------------------------------------------------------
# 2. Function: Add Candidates to ChromaDB (With Batching)
# ---------------------------------------------------------
def add_candidates_to_chroma(json_data, batch_size=100):
    """
    Parses candidate JSON and stores embeddings in ChromaDB.
    Uses batch processing to handle massive datasets smoothly without crashing RAM.
    """
    total_candidates = len(json_data)
    print(f"Preparing to process {total_candidates} candidates...")
    
    # Process the dataset in chunks/batches to protect memory
    for i in range(0, total_candidates, batch_size):
        batch = json_data[i : i + batch_size]
        
        documents = []
        metadatas = []
        ids = []
        
        for candidate in batch:
            cand_id = candidate.get("candidate_id")
            profile = candidate.get("profile", {})
            
            # Extract text features
            headline = profile.get("headline", "")
            summary = profile.get("summary", "")
            
            # Safely compile lists of skills and past experiences
            skills_list = [skill.get("name") for skill in candidate.get("skills", []) if skill.get("name")]
            skills_text = ", ".join(skills_list)
            
            experience_texts = [exp.get("description", "") for exp in candidate.get("career_history", []) if exp.get("description")]
            experience_summary = " ".join(experience_texts)
            
            # Build the final text document that will be converted to numbers (embedded)
            text_profile = f"Headline: {headline}. Summary: {summary}. Skills: {skills_text}. Experience: {experience_summary}."
            
            documents.append(text_profile)
            ids.append(cand_id)
            
            # Store metadata for quick filtering later
            metadatas.append({
                "name": profile.get("anonymized_name", "Unknown"),
                "title": profile.get("current_title", "Unknown"),
                "location": profile.get("location", "Unknown"),
                "years_of_experience": profile.get("years_of_experience", 0.0)
            })
            
        print(f"Generating embeddings for batch {i+1} to {min(i + batch_size, total_candidates)}...")
        embeddings = model.encode(documents).tolist()
        
        # Insert the batch into the ChromaDB collection
        collection.add(
            documents=documents,
            embeddings=embeddings,
            metadatas=metadatas,
            ids=ids
        )
        
    print(f"\nSuccessfully added all {total_candidates} candidates to the database!")

# ---------------------------------------------------------
# 3. Function: Query the Collection
# ---------------------------------------------------------
def search_candidates(query_string, top_k=50):
    """
    Takes a natural language search string, embeds it, and queries 
    ChromaDB for the most semantically similar candidate profiles.
    """
    query_embedding = model.encode([query_string]).tolist()
    
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=top_k
    )
    
    return results

# ---------------------------------------------------------
# 4. Execution Pipeline
# ---------------------------------------------------------
if __name__ == "__main__":
    try:
        # Load the JSON data
        with open('sample_candidates.json', 'r') as file:
            candidate_data = json.load(file)
            
        # Add the parsed data to our Chroma collection
        add_candidates_to_chroma(candidate_data)
        
        # Define a test search query based on skills in your dataset
        search_query = "Looking for a backend data engineer experienced with Spark, Airflow, and Python data pipelines."
        print(f"\nSearching for: '{search_query}'\n")
        
        # Execute the search function (fetching top 10 for readability)
        results = search_candidates(search_query, top_k=10)
        
        # Display the Top Matches
        print("--- Top Candidate Matches ---")
        for i in range(len(results['ids'][0])):
            candidate_id = results['ids'][0][i]
            metadata = results['metadatas'][0][i]
            distance = results['distances'][0][i] 
            
            print(f"Rank {i+1}: {metadata['name']} ({candidate_id})")
            print(f"Title: {metadata['title']} | Exp: {metadata['years_of_experience']} years")
            print(f"Distance Score: {distance:.4f} (Lower = better match)")
            print("-" * 40)
            
    except FileNotFoundError:
        print("Error: 'sample_candidates.json' not found. Please ensure it is in the same directory as this notebook.")

ModuleNotFoundError: No module named 'chromadb'